In [ ]:
# ============================================================
# IEEE-CIS FRAUD DETECTION
# LEAKAGE-SAFE LIGHTGBM BASELINE
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import gc
import warnings

warnings.filterwarnings("ignore")


# ============================================================
# 0. CONFIG
# ============================================================

TRAIN_TRANSACTION_PATH = "/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv"
TRAIN_IDENTITY_PATH    = "/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv"

RANDOM_STATE = 42
TRAIN_RATIO = 0.80

# Baseline leakage-safe
SCALE_POS_WEIGHT = 1.0


# ============================================================
# 1. LOAD DATA
# ============================================================

print("=" * 70)
print("1. Loading data...")
print("=" * 70)

train_transaction = pd.read_csv(
    TRAIN_TRANSACTION_PATH
)

train_identity = pd.read_csv(
    TRAIN_IDENTITY_PATH
)

print(
    f"train_transaction : {train_transaction.shape}"
)

print(
    f"train_identity    : {train_identity.shape}"
)

# ============================================================
# 2. MERGE TRANSACTION + IDENTITY
# ============================================================

print("\n" + "=" * 70)
print("2. Merge transaction + identity...")
print("=" * 70)

train_tx = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(f"Shape setelah merge: {train_tx.shape}")

del train_transaction
del train_identity

gc.collect()


# ============================================================
# 3. DOWNCASTING
# ============================================================

print("\n" + "=" * 70)
print("3. Downcasting...")
print("=" * 70)

memory_before = train_tx.memory_usage(deep=True).sum() / 1024**2


def downcast_dataframe(df):
    """
    Mengurangi penggunaan RAM tanpa mengubah informasi
    secara material.
    """

    for col in df.columns:

        col_type = df[col].dtype

        # Integer
        if pd.api.types.is_integer_dtype(col_type):

            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)

            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)

            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)

            else:
                df[col] = df[col].astype(np.int64)

        # Float
        elif pd.api.types.is_float_dtype(col_type):

            df[col] = pd.to_numeric(
                df[col],
                downcast="float"
            )

    return df


train_tx = downcast_dataframe(train_tx)

memory_after = train_tx.memory_usage(deep=True).sum() / 1024**2

print(
    f"Mem usage: "
    f"{memory_before:.1f} → {memory_after:.1f} MB"
)


# ============================================================
# 4. MISSING INDICATORS + IMPUTATION
# ============================================================

print("\n" + "=" * 70)
print("4. Missing Indicators + Imputasi...")
print("=" * 70)

# Simpan kolom object sebelum encoding
object_cols = train_tx.select_dtypes(
    include=["object"]
).columns.tolist()

print(f"Jumlah object columns: {len(object_cols)}")


# ------------------------------------------------------------
# Missing indicators
# ------------------------------------------------------------

for col in train_tx.columns:

    if train_tx[col].isna().any():

        train_tx[f"{col}_missing"] = (
            train_tx[col].isna().astype(np.int8)
        )


# ------------------------------------------------------------
# Imputation
# ------------------------------------------------------------

for col in train_tx.columns:

    if train_tx[col].isna().any():

        if pd.api.types.is_numeric_dtype(train_tx[col]):

            # Median lebih robust terhadap outlier
            median_value = train_tx[col].median()

            if pd.isna(median_value):
                median_value = 0

            train_tx[col] = train_tx[col].fillna(
                median_value
            )

        else:

            train_tx[col] = train_tx[col].fillna(
                "__MISSING__"
            )


print(
    "Sisa missing values:",
    train_tx.isna().sum().sum()
)


# ============================================================
# 5. LABEL ENCODING
# ============================================================

print("\n" + "=" * 70)
print("5. Label Encoding...")
print("=" * 70)

object_cols = train_tx.select_dtypes(
    include=["object"]
).columns.tolist()

print(f"Object columns sebelum encoding: {len(object_cols)}")


for col in object_cols:

    # category codes
    train_tx[col] = (
        train_tx[col]
        .astype("category")
        .cat.codes
        .astype(np.int32)
    )


print(
    f"Shape setelah preprocessing: {train_tx.shape}"
)

print(
    "Jumlah kolom object tersisa:",
    train_tx.select_dtypes(include=["object"]).shape[1]
)


gc.collect()


# ============================================================
# 6. TIME-BASED SPLIT
# ============================================================

print("\n" + "=" * 70)
print("6. Time-based Split")
print("=" * 70)

# WAJIB:
# Sort berdasarkan TransactionDT sebelum split.
train_tx = train_tx.sort_values(
    "TransactionDT"
).reset_index(drop=True)


split_point = int(
    len(train_tx) * TRAIN_RATIO
)

train_df = train_tx.iloc[
    :split_point
].copy()

val_df = train_tx.iloc[
    split_point:
].copy()


print(
    f"Train: {train_df.shape} | "
    f"Fraud rate: {train_df['isFraud'].mean() * 100:.2f}%"
)

print(
    f"Val  : {val_df.shape} | "
    f"Fraud rate: {val_df['isFraud'].mean() * 100:.2f}%"
)


# Setelah split, train_tx tidak diperlukan lagi.
del train_tx

gc.collect()


# ============================================================
# 7. SAFE FEATURE ENGINEERING
# ============================================================

print("\n" + "=" * 70)
print("7. Feature Engineering (NO LEAKAGE)")
print("=" * 70)


# ============================================================
# 7A. STATIC FEATURES
# ============================================================

print("\n[7A] Static transaction features...")


for df in [train_df, val_df]:

    # Log amount
    df["amt_log"] = np.log1p(
        df["TransactionAmt"]
    )

    # Decimal component
    df["amt_decimal"] = (
        (
            df["TransactionAmt"]
            - df["TransactionAmt"].astype(int)
        ) * 1000
    ).round().astype(np.int32)

    # Hour
    df["hour"] = (
        df["TransactionDT"] // 3600
    ) % 24

    # Day of week
    df["dayofweek"] = (
        df["TransactionDT"] // (3600 * 24)
    ) % 7

    # Night indicator
    df["is_night"] = (
        df["hour"].isin(
            [0, 1, 2, 3, 4, 5]
        )
        .astype(np.int8)
    )


# ============================================================
# 7B. TRANSACTIONDT NORMALIZATION
# ============================================================

print("\n[7B] TransactionDT normalization...")


# PENTING:
# max hanya berasal dari TRAIN.
max_dt_train = train_df["TransactionDT"].max()

train_df["TransactionDT_norm"] = (
    train_df["TransactionDT"]
    / max_dt_train
)

val_df["TransactionDT_norm"] = (
    val_df["TransactionDT"]
    / max_dt_train
)


# ============================================================
# 7C. FREQUENCY ENCODING
# ============================================================

print("\n[7C] Frequency Encoding...")


high_card_cols = [
    "card1",
    "card2",
    "card3",
    "card5",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceInfo"
]


for col in high_card_cols:

    if col not in train_df.columns:
        continue

    print(f"  → {col}")

    # --------------------------------------------------------
    # CRITICAL:
    # frequency statistics HANYA dari TRAIN
    # --------------------------------------------------------

    freq = train_df[
        col
    ].value_counts(
        dropna=False
    )

    train_df[f"{col}_freq"] = (
        train_df[col]
        .map(freq)
        .fillna(1)
        .astype(np.float32)
    )

    val_df[f"{col}_freq"] = (
        val_df[col]
        .map(freq)
        .fillna(1)
        .astype(np.float32)
    )


# ============================================================
# 7D. CARD1 AGGREGATION
# ============================================================

print("\n[7D] Aggregasi card1...")


card1_agg = (
    train_df
    .groupby("card1")["TransactionAmt"]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max",
            "count"
        ]
    )
    .reset_index()
)


card1_agg.columns = [
    "card1",
    "card1_amt_mean",
    "card1_amt_std",
    "card1_amt_min",
    "card1_amt_max",
    "card1_count"
]


# ------------------------------------------------------------
# Apply train-derived statistics
# ------------------------------------------------------------

train_df = train_df.merge(
    card1_agg,
    on="card1",
    how="left"
)

val_df = val_df.merge(
    card1_agg,
    on="card1",
    how="left"
)


# ------------------------------------------------------------
# Amount relative to card1 historical mean
# ------------------------------------------------------------

train_df[
    "TransactionAmt_to_card1_mean"
] = (
    train_df["TransactionAmt"]
    /
    (train_df["card1_amt_mean"] + 1e-3)
)


val_df[
    "TransactionAmt_to_card1_mean"
] = (
    val_df["TransactionAmt"]
    /
    (val_df["card1_amt_mean"] + 1e-3)
)


del card1_agg

gc.collect()


# ============================================================
# 7E. UID FEATURES
# ============================================================

print("\n[7E] UID Features...")


train_df["uid"] = (
    train_df["card1"].astype(str)
    + "_"
    + train_df["addr1"].astype(str)
    + "_"
    + train_df["P_emaildomain"].astype(str)
)


val_df["uid"] = (
    val_df["card1"].astype(str)
    + "_"
    + val_df["addr1"].astype(str)
    + "_"
    + val_df["P_emaildomain"].astype(str)
)


# ------------------------------------------------------------
# UID statistics — TRAIN ONLY
# ------------------------------------------------------------

uid_agg = (
    train_df
    .groupby("uid")["TransactionAmt"]
    .agg(
        [
            "count",
            "mean"
        ]
    )
    .reset_index()
)


uid_agg.columns = [
    "uid",
    "uid_count",
    "uid_amt_mean"
]


# ------------------------------------------------------------
# Merge train-derived UID statistics
# ------------------------------------------------------------

train_df = train_df.merge(
    uid_agg,
    on="uid",
    how="left"
)

val_df = val_df.merge(
    uid_agg,
    on="uid",
    how="left"
)


# ------------------------------------------------------------
# Amount relative to UID mean
# ------------------------------------------------------------

train_df[
    "uid_amt_ratio"
] = (
    train_df["TransactionAmt"]
    /
    (train_df["uid_amt_mean"] + 1e-3)
)


val_df[
    "uid_amt_ratio"
] = (
    val_df["TransactionAmt"]
    /
    (val_df["uid_amt_mean"] + 1e-3)
)


# UID tidak dibutuhkan lagi setelah aggregation.
train_df.drop(
    columns=["uid"],
    inplace=True
)

val_df.drop(
    columns=["uid"],
    inplace=True
)


del uid_agg

gc.collect()


# ============================================================
# 7F. FILL AGGREGATION NaN
# ============================================================

print("\n[7F] Filling aggregation NaN...")


agg_keywords = [
    "_mean",
    "_std",
    "_count",
    "_min",
    "_max",
    "_ratio",
    "_freq"
]


agg_cols = [
    col
    for col in train_df.columns
    if any(
        keyword in col
        for keyword in agg_keywords
    )
]


for col in agg_cols:

    if col in train_df.columns:
        train_df[col] = (
            train_df[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
        )

    if col in val_df.columns:
        val_df[col] = (
            val_df[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
        )


# ============================================================
# 8. DROP HIGH-CARDINALITY COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("8. Drop high-cardinality columns...")
print("=" * 70)


# Jangan drop fitur frequency encoding.
# Yang dibuang adalah raw high-cardinality categorical
# yang sudah direpresentasikan melalui encoding statistik.

drop_high_card_cols = [
    "DeviceInfo"
]


for col in drop_high_card_cols:

    if col in train_df.columns:
        train_df.drop(
            columns=[col],
            inplace=True
        )

    if col in val_df.columns:
        val_df.drop(
            columns=[col],
            inplace=True
        )


# ============================================================
# 9. FINAL DATASET
# ============================================================

print("\n" + "=" * 70)
print("9. Preparing final dataset...")
print("=" * 70)


drop_cols = [
    "isFraud",
    "TransactionID",
    "TransactionDT"
]


# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

y_train = train_df["isFraud"].astype(np.int8)
y_val = val_df["isFraud"].astype(np.int8)


# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

x_train = train_df.drop(
    columns=drop_cols
)

x_val = val_df.drop(
    columns=drop_cols
)


print(
    f"Final shape → "
    f"Train: {train_df.shape} | "
    f"Val: {val_df.shape}"
)

print(
    f"x_train: {x_train.shape} | "
    f"x_val: {x_val.shape}"
)


# ============================================================
# 10. SAFETY CHECK
# ============================================================

print("\n" + "=" * 70)
print("10. Safety checks...")
print("=" * 70)


# Pastikan tidak ada object
object_train = x_train.select_dtypes(
    include=["object"]
).shape[1]

object_val = x_val.select_dtypes(
    include=["object"]
).shape[1]


print(
    f"Object columns train: {object_train}"
)

print(
    f"Object columns val  : {object_val}"
)


# Pastikan jumlah feature sama
assert x_train.shape[1] == x_val.shape[1], (
    "Jumlah feature train dan validation berbeda!"
)


# Pastikan nama feature sama
assert list(x_train.columns) == list(x_val.columns), (
    "Nama feature train dan validation berbeda!"
)


# Pastikan tidak ada infinite
assert np.isfinite(
    x_train.select_dtypes(include=np.number)
).all().all(), (
    "Ada nilai infinity di x_train!"
)

assert np.isfinite(
    x_val.select_dtypes(include=np.number)
).all().all(), (
    "Ada nilai infinity di x_val!"
)


print("✓ Feature consistency: OK")
print("✓ Object columns: OK")
print("✓ Infinite values: OK")


# ============================================================
# 11. LIGHTGBM
# ============================================================

print("\n" + "=" * 70)
print("11. Training LightGBM")
print("=" * 70)


print(
    f"scale_pos_weight: {SCALE_POS_WEIGHT}"
)


model = lgb.LGBMClassifier(

    n_estimators=1000,

    learning_rate=0.05,

    num_leaves=64,

    max_depth=-1,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=SCALE_POS_WEIGHT,

    random_state=RANDOM_STATE,

    n_jobs=-1,

    importance_type="gain"
)


model.fit(

    x_train,

    y_train,

    eval_set=[
        (x_val, y_val)
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=40,
            verbose=False
        )
    ]
)


print(
    f"Best iteration: "
    f"{model.best_iteration_}"
)


# ============================================================
# 12. PREDICTION
# ============================================================

print("\n" + "=" * 70)
print("12. Prediction...")
print("=" * 70)


y_pred_proba = model.predict_proba(
    x_val
)[:, 1]


# ============================================================
# 13. GLOBAL METRICS
# ============================================================

print("\n" + "=" * 70)
print("13. Hasil Evaluasi")
print("=" * 70)


roc_auc = roc_auc_score(
    y_val,
    y_pred_proba
)

pr_auc = average_precision_score(
    y_val,
    y_pred_proba
)


print(
    f"ROC-AUC : {roc_auc:.4f}"
)

print(
    f"PR-AUC  : {pr_auc:.4f}"
)


# ============================================================
# 14. THRESHOLD ANALYSIS
# ============================================================

print("\nThreshold Analysis:")

print(
    f"{'Th':<7}"
    f"{'Precision':<12}"
    f"{'Recall':<11}"
    f"{'F1':<11}"
    f"{'FP':<8}"
    f"{'FN':<8}"
)

print("-" * 60)


thresholds = [
    0.10,
    0.12,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35
]


for threshold in thresholds:

    y_pred = (
        y_pred_proba >= threshold
    ).astype(int)


    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )


    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred
    ).ravel()


    print(
        f"{threshold:<7.2f}"
        f"{precision:<12.4f}"
        f"{recall:<11.4f}"
        f"{f1:<11.4f}"
        f"{fp:<8}"
        f"{fn:<8}"
    )


# ============================================================
# 15. FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("15. 20 Fitur Paling Berpengaruh")
print("=" * 70)


feature_importance = pd.DataFrame({

    "feature": x_train.columns,

    "gain": model.feature_importances_

})


feature_importance = (
    feature_importance
    .sort_values(
        "gain",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    feature_importance.head(20).to_string(
        index=False
    )
)


# ============================================================
# 16. NEW FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("16. Fitur Baru yang Masuk")
print("=" * 70)


new_features = [

    "card1_amt_std",
    "card1_amt_mean",
    "card1_amt_min",
    "card1_amt_max",
    "card1_count",

    "uid_amt_mean",
    "uid_count",

    "TransactionAmt_to_card1_mean",
    "uid_amt_ratio",

    "amt_log",
    "amt_decimal",

    "hour",
    "dayofweek",
    "is_night",

    "TransactionDT_norm"
]


new_feature_importance = (
    feature_importance[
        feature_importance["feature"].isin(
            new_features
        )
    ]
    .sort_values(
        "gain",
        ascending=False
    )
)


print(
    new_feature_importance.to_string(
        index=False
    )
)


# ============================================================
# 17. SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("17. FINAL SUMMARY")
print("=" * 70)


print(
    f"Train samples : {len(x_train):,}"
)

print(
    f"Val samples   : {len(x_val):,}"
)

print(
    f"Train fraud   : {y_train.sum():,} "
    f"({y_train.mean()*100:.2f}%)"
)

print(
    f"Val fraud     : {y_val.sum():,} "
    f"({y_val.mean()*100:.2f}%)"
)

print(
    f"Features      : {x_train.shape[1]:,}"
)

print(
    f"Best iteration: {model.best_iteration_}"
)

print(
    f"ROC-AUC       : {roc_auc:.4f}"
)

print(
    f"PR-AUC        : {pr_auc:.4f}"
)

print(
    f"Scale weight  : {SCALE_POS_WEIGHT}"
)

print("\n✅ Pipeline selesai.")

In [ ]:
# ============================================================
# 18. SHAP INTERPRETABILITY
# ============================================================

print("\n" + "=" * 70)
print("18. SHAP FEATURE IMPORTANCE")
print("=" * 70)

import shap

# ------------------------------------------------------------
# 18A. Ambil sample validation
# ------------------------------------------------------------
# SHAP pada seluruh 118k transaksi bisa cukup berat.
# Kita gunakan sample 5,000 transaksi untuk interpretasi.

SHAP_SAMPLE_SIZE = 5000

if len(x_val) > SHAP_SAMPLE_SIZE:
    x_shap = x_val.sample(
        n=SHAP_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )
else:
    x_shap = x_val.copy()

print(
    f"SHAP sample: {x_shap.shape}"
)


# ------------------------------------------------------------
# 18B. SHAP TreeExplainer
# ------------------------------------------------------------

explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(
    x_shap
)


# ------------------------------------------------------------
# 18C. Handle LightGBM binary output
# ------------------------------------------------------------

if isinstance(shap_values, list):

    # Binary classification:
    # class 1 = fraud
    shap_values_fraud = shap_values[1]

else:

    shap_values_fraud = shap_values


print(
    f"SHAP values shape: {shap_values_fraud.shape}"
)


# ============================================================
# 18D. GLOBAL SHAP IMPORTANCE
# ============================================================

shap_importance = pd.DataFrame({

    "feature": x_shap.columns,

    "mean_abs_shap":
        np.abs(shap_values_fraud).mean(axis=0)

})


shap_importance = (
    shap_importance
    .sort_values(
        "mean_abs_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n20 Fitur Paling Berpengaruh Menurut SHAP:")

print(
    shap_importance.head(20).to_string(
        index=False
    )
)


# ============================================================
# 18E. SHAP SUMMARY PLOT
# ============================================================

print("\nMembuat SHAP Summary Plot...")

shap.summary_plot(
    shap_values_fraud,
    x_shap,
    max_display=20
)

In [ ]:
# ============================================================
# 19. SHAP INDIVIDUAL EXPLANATION (Fixed)
# ============================================================

import shap
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("19. SHAP INDIVIDUAL EXPLANATION")
print("=" * 70)

# ------------------------------------------------------------
# Important: We will compute SHAP only on selected transactions
# ------------------------------------------------------------

pred_proba = model.predict_proba(x_val)[:, 1]

# Get top fraud and non-fraud indices from full validation set
top_fraud_idx = np.argsort(pred_proba)[-5:][::-1]
top_nonfraud_idx = np.argsort(pred_proba)[:5]

print("\nTop predicted FRAUD transactions:")
for i in top_fraud_idx:
    print(f"  Index: {i:6d} | Pred Proba: {pred_proba[i]:.4f} | Actual: {y_val.iloc[i]}")

print("\nTop predicted NON-FRAUD transactions:")
for i in top_nonfraud_idx:
    print(f"  Index: {i:6d} | Pred Proba: {pred_proba[i]:.4f} | Actual: {y_val.iloc[i]}")

# Select transactions we want to explain
selected_idx = list(top_fraud_idx[:3]) + list(top_nonfraud_idx[:3])
X_explain = x_val.iloc[selected_idx]

print(f"\nComputing SHAP values for {len(selected_idx)} selected transactions...")

# Create explainer and compute SHAP only for these rows
explainer = shap.TreeExplainer(model)
shap_values_selected = explainer.shap_values(X_explain)

# ------------------------------------------------------------
# 1. Force Plot - Single transaction (highest fraud probability)
# ------------------------------------------------------------
sample_pos = 0   # first one in selected_idx (highest fraud)

print(f"\n>>> Force Plot for transaction index: {selected_idx[sample_pos]}")
print(f"    Predicted Fraud Probability : {pred_proba[selected_idx[sample_pos]]:.4f}")
print(f"    Actual Label                : {y_val.iloc[selected_idx[sample_pos]]}")

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values_selected[sample_pos],
    X_explain.iloc[sample_pos],
    matplotlib=False
)

# ------------------------------------------------------------
# 2. Waterfall Plot
# ------------------------------------------------------------
print("\n>>> Waterfall Plot")

explanation = shap.Explanation(
    values=shap_values_selected[sample_pos],
    base_values=explainer.expected_value,
    data=X_explain.iloc[sample_pos].values,
    feature_names=X_explain.columns.tolist()
)

plt.figure(figsize=(10, 8))
shap.plots.waterfall(explanation, max_display=15, show=False)
plt.title(
    f"Waterfall Plot - Transaction #{selected_idx[sample_pos]}\n"
    f"Predicted Fraud: {pred_proba[selected_idx[sample_pos]]:.4f} | "
    f"Actual: {y_val.iloc[selected_idx[sample_pos]]}",
    fontsize=12
)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3. Force Plot for multiple transactions
# ------------------------------------------------------------
print("\n>>> Force Plot for multiple transactions (3 Fraud + 3 Non-Fraud)")

shap.force_plot(
    explainer.expected_value,
    shap_values_selected,
    X_explain,
    matplotlib=False
)

In [ ]:
# ============================================================
# 20. SHAP DIRECTION SUMMARY (Fixed)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

print("=" * 70)
print("20. SHAP DIRECTION SUMMARY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Recompute SHAP on a sample (to avoid memory issues)
# ------------------------------------------------------------
sample_size = 3000
shap_sample = x_val.sample(sample_size, random_state=42)

print(f"Computing SHAP values on {sample_size} samples...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(shap_sample)

print("SHAP values shape:", shap_values.shape)

# ------------------------------------------------------------
# 2. Calculate direction
# ------------------------------------------------------------
mean_shap = np.mean(shap_values, axis=0)
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

direction_df = pd.DataFrame({
    'feature': shap_sample.columns,
    'mean_shap': mean_shap,
    'mean_abs_shap': mean_abs_shap
})

direction_df['direction'] = direction_df['mean_shap'].apply(
    lambda x: 'Increases Fraud Risk' if x > 0 else 'Decreases Fraud Risk'
)

direction_df = direction_df.sort_values('mean_abs_shap', ascending=False)

print("\nTop 20 Features - Direction Summary:")
print(direction_df.head(20).to_string(index=False))

# ------------------------------------------------------------
# 3. Visualization
# ------------------------------------------------------------
top_n = 20
plot_df = direction_df.head(top_n).copy()
plot_df = plot_df.sort_values('mean_shap')

plt.figure(figsize=(12, 8))
colors = ['#d62728' if x > 0 else '#1f77b4' for x in plot_df['mean_shap']]

plt.barh(plot_df['feature'], plot_df['mean_shap'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Mean SHAP Value (Direction)')
plt.title('SHAP Direction - Top 20 Features\n(Red = Increases Fraud Risk | Blue = Decreases Fraud Risk)')
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------
print("\n=== Direction Count ===")
print(direction_df['direction'].value_counts())

print("\n=== Top features that INCREASE fraud risk ===")
print(direction_df[direction_df['direction'] == 'Increases Fraud Risk']
      .head(10)[['feature', 'mean_shap', 'mean_abs_shap']]
      .to_string(index=False))

print("\n=== Top features that DECREASE fraud risk ===")
print(direction_df[direction_df['direction'] == 'Decreases Fraud Risk']
      .head(10)[['feature', 'mean_shap', 'mean_abs_shap']]
      .to_string(index=False))

In [ ]:
# ============================================================
# 21. FINAL PIPELINE (BEST VERSION - FIXED)
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import gc
from sklearn.preprocessing import LabelEncoder

print("=" * 70)
print("21. FINAL PIPELINE - TRAIN + TEST + SUBMISSION")
print("=" * 70)

PATH = '/kaggle/input/competitions/ieee-fraud-detection/'

# ------------------------------------------------------------
# Helper Functions
# ------------------------------------------------------------
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and col_type.name != 'category':
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    return df


def create_basic_features(df):
    df = df.copy()
    df['amt_log'] = np.log1p(df['TransactionAmt'])
    df['amt_decimal'] = ((df['TransactionAmt'] - df['TransactionAmt'].astype(int)) * 1000).round().astype(np.int16)
    df['hour'] = ((df['TransactionDT'] // 3600) % 24).astype(np.int8)
    df['dayofweek'] = ((df['TransactionDT'] // (3600 * 24)) % 7).astype(np.int8)
    df['is_night'] = df['hour'].isin([0, 1, 2, 3, 4, 5]).astype(np.int8)
    return df


def apply_frequency_encoding(train, test, cols):
    for col in cols:
        if col in train.columns:
            freq = train[col].value_counts(dropna=False)
            train[f'{col}_freq'] = train[col].map(freq).astype('float32').fillna(1)
            test[f'{col}_freq']  = test[col].map(freq).astype('float32').fillna(1)
    return train, test


def apply_uid_features(train, test):
    for df in [train, test]:
        df['uid'] = (
            df['card1'].astype(str) + '_' +
            df['addr1'].astype(str) + '_' +
            df['P_emaildomain'].astype(str)
        )
    
    uid_agg = train.groupby('uid')['TransactionAmt'].agg(['count', 'mean']).reset_index()
    uid_agg.columns = ['uid', 'uid_count', 'uid_amt_mean']
    
    train = train.merge(uid_agg, on='uid', how='left')
    test  = test.merge(uid_agg, on='uid', how='left')
    
    train['uid_amt_ratio'] = train['TransactionAmt'] / (train['uid_amt_mean'] + 1e-3)
    test['uid_amt_ratio']  = test['TransactionAmt'] / (test['uid_amt_mean'] + 1e-3)
    
    train.drop(columns=['uid'], inplace=True)
    test.drop(columns=['uid'], inplace=True)
    
    return train, test


def apply_card1_agg(train, test):
    card1_agg = train.groupby('card1')['TransactionAmt'].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
    card1_agg.columns = ['card1', 'card1_amt_mean', 'card1_amt_std', 'card1_amt_min', 'card1_amt_max', 'card1_count']
    
    train = train.merge(card1_agg, on='card1', how='left')
    test  = test.merge(card1_agg, on='card1', how='left')
    
    train['TransactionAmt_to_card1_mean'] = train['TransactionAmt'] / (train['card1_amt_mean'] + 1e-3)
    test['TransactionAmt_to_card1_mean']  = test['TransactionAmt'] / (test['card1_amt_mean'] + 1e-3)
    
    return train, test


# ------------------------------------------------------------
# 1. Load Data
# ------------------------------------------------------------
print("1. Loading data...")

train_transaction = pd.read_csv(PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(PATH + 'train_identity.csv')
test_transaction  = pd.read_csv(PATH + 'test_transaction.csv')
test_identity     = pd.read_csv(PATH + 'test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity, on='TransactionID', how='left')

# Fix column names in test (id-01 → id_01)
test.columns = [c.replace('-', '_') for c in test.columns]

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")

del train_transaction, train_identity, test_transaction, test_identity
gc.collect()

# ------------------------------------------------------------
# 2. Basic Preprocessing
# ------------------------------------------------------------
print("2. Basic preprocessing...")

train = reduce_mem_usage(train)
test  = reduce_mem_usage(test)

# Missing indicators
original_cols = [c for c in train.columns if c != 'isFraud']
cols_with_missing = [c for c in original_cols if train[c].isnull().any()]

missing_train = {f'{c}_is_missing': train[c].isnull().astype(np.int8) for c in cols_with_missing}
missing_test  = {f'{c}_is_missing': test[c].isnull().astype(np.int8) for c in cols_with_missing}

train = pd.concat([train, pd.DataFrame(missing_train, index=train.index)], axis=1)
test  = pd.concat([test, pd.DataFrame(missing_test, index=test.index)], axis=1)

# Imputation
numerical_cols = [c for c in original_cols if train[c].dtype != 'object']
categorical_cols = [c for c in original_cols if train[c].dtype == 'object']

for col in numerical_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col]  = test[col].fillna(median_val)

for col in categorical_cols:
    train[col] = train[col].fillna('Unknown')
    test[col]  = test[col].fillna('Unknown')

# Label Encoding
for col in categorical_cols:
    le = LabelEncoder()
    le.fit(list(train[col].astype(str)) + list(test[col].astype(str)))
    train[col] = le.transform(train[col].astype(str)).astype(np.int32)
    test[col]  = le.transform(test[col].astype(str)).astype(np.int32)

# ------------------------------------------------------------
# 3. Feature Engineering (no leakage)
# ------------------------------------------------------------
print("3. Feature Engineering...")

train = create_basic_features(train)
test  = create_basic_features(test)

# TransactionDT_norm
max_dt = train['TransactionDT'].max()
train['TransactionDT_norm'] = train['TransactionDT'] / max_dt
test['TransactionDT_norm']  = test['TransactionDT'] / max_dt

high_card_cols = [
    'card1', 'card2', 'card3', 'card5',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
    'DeviceInfo'
]

train, test = apply_frequency_encoding(train, test, high_card_cols)
train, test = apply_uid_features(train, test)
train, test = apply_card1_agg(train, test)

# Drop high cardinality original columns
cols_to_drop = [c for c in high_card_cols if c in train.columns]
train.drop(columns=cols_to_drop, inplace=True)
test.drop(columns=cols_to_drop, inplace=True)

# Fill remaining NaN in aggregated features
agg_cols = [c for c in train.columns if any(x in c for x in ['_mean', '_std', '_count', '_min', '_max', '_ratio', '_freq'])]
for col in agg_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

# ------------------------------------------------------------
# 4. Prepare data for modeling
# ------------------------------------------------------------
print("4. Preparing data...")

drop_cols = ['isFraud', 'TransactionID', 'TransactionDT']
features = [c for c in train.columns if c not in drop_cols]

X_train = train[features]
y_train = train['isFraud']
X_test  = test[features]

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

# ------------------------------------------------------------
# 5. Train Final Model
# ------------------------------------------------------------
print("5. Training final model...")

scale_pos_weight = np.sqrt((y_train == 0).sum() / (y_train == 1).sum())
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

final_model = lgb.LGBMClassifier(
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.7,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbose=-1
)

final_model.fit(X_train, y_train)
print("Training completed.")

# ------------------------------------------------------------
# 6. Predict & Create Submission
# ------------------------------------------------------------
print("6. Creating submission...")

test_pred = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test['TransactionID'],
    'isFraud': test_pred
})

submission.to_csv('submission.csv', index=False)
print("Saved: submission.csv")
print(submission.head())

# ------------------------------------------------------------
# 7. Save Model
# ------------------------------------------------------------
joblib.dump(final_model, 'final_lgbm_fraud_model.pkl')
print("Saved: final_lgbm_fraud_model.pkl")

print("\n✅ Final pipeline completed successfully!")